# Lab 8: Spark Structured Streaming + Kafka

## Cel

- Połączenie Spark Structured Streaming z Apache Kafka,
- Odczyt strumienia z tematu Kafka,
- Transformacje i agregacje na danych z Kafki,
- Zapis wyników do konsoli i do Kafki.

---

## Przygotowanie

1. Upewnij się, że Kafka działa (`docker compose up -d`).
2. Zainstaluj pakiet Spark-Kafka:
```bash
pip install pyspark
```

Spark wymaga dodatkowego JAR do połączenia z Kafką. Można go dodać przy tworzeniu sesji.

## Część 1: Odczyt z Kafki

### Zadanie 1.1 — Połącz Sparka z Kafką

In [ ]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
    .appName("Lab8-Kafka")
    .master("local[*]")
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Spark + Kafka ready")

### Zadanie 1.2 — Odczytaj strumień z tematu Kafka

Utwórz temat `spark-test` (jeśli nie istnieje):
```bash
docker compose exec kafka kafka-topics --create \
  --topic spark-test --bootstrap-server localhost:9092 \
  --partitions 3 --replication-factor 1
```

Odczytaj strumień:

In [ ]:
kafka_df = (spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "localhost:9092")
    .option("subscribe", "spark-test")
    .option("startingOffsets", "earliest")
    .load()
)

# Sprawdz schemat - Kafka zwraca kolumny: key, value, topic, partition, offset, timestamp
kafka_df.printSchema()

### Zadanie 1.3 — Dekoduj dane

Kolumna `value` w Kafce jest typu binary. Musisz ją przekonwertować na string, a następnie sparsować JSON.

Napisz transformację, która:
1. Konwertuje `value` na string
2. Parsuje JSON (użyj `from_json` z odpowiednim schematem)
3. Wyciąga pola: `id`, `kwota`, `sklep`, `timestamp`

In [ ]:
from pyspark.sql.functions import col, from_json, expr
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType

# Schemat danych JSON
schema = StructType([
    StructField("id", StringType()),
    StructField("kwota", DoubleType()),
    StructField("sklep", StringType()),
    StructField("timestamp", StringType()),
])

# TWÓJ KOD
# parsed = kafka_df.select(
#     from_json(col("value").cast("string"), schema).alias("data")
# ).select("data.*")
# parsed powinien miec kolumny: id, kwota, sklep, timestamp

### Zadanie 1.4 — Uruchom producenta i strumień

W osobnym terminalu uruchom producenta z Lab 4 (lub napisz nowego), który wysyła transakcje do tematu `spark-test`.

Następnie uruchom strumień:

In [ ]:
batch_counter = {"count": 0}
def process_batch(df, batch_id):
    batch_counter["count"] += 1
    print(f"--- Batch {batch_id} ---")
    df.show(truncate=False)
    if batch_counter["count"] >= 5:
        raise Exception("Stop")

# TWÓJ KOD
# query = parsed.writeStream.format("console").foreachBatch(process_batch).start()
# try:
#     query.awaitTermination()
# except:
#     query.stop()

---

## Część 2: Agregacje na strumieniu z Kafki

### Zadanie 2.1 — Suma transakcji per sklep (complete mode)

In [ ]:
from pyspark.sql.functions import sum as spark_sum, count

# TWÓJ KOD
# agg = parsed.groupBy("sklep").agg(
#     count("*").alias("liczba"),
#     spark_sum("kwota").alias("suma")
# )
# Uruchom w outputMode("complete")

### Zadanie 2.2 — Okno czasowe + watermark

Pogrupuj transakcje w okna 1-minutowe z watermarkiem 30 sekund.
Oblicz sumę i średnią kwotę per okno.

In [ ]:
from pyspark.sql.functions import window, avg, to_timestamp

# TWÓJ KOD
# Pamietaj o skonwertowaniu kolumny timestamp na typ timestamp
# parsed_ts = parsed.withColumn("ts", to_timestamp("timestamp"))
# windowed = parsed_ts.withWatermark("ts", "30 seconds").groupBy(window("ts", "1 minute"))...

---

## Część 3: Zapis wyników do Kafki

### Zadanie 3.1 — Filtruj i wysyłaj alerty do nowego tematu

Utwórz temat `alerts`:
```bash
docker compose exec kafka kafka-topics --create \
  --topic alerts --bootstrap-server localhost:9092 \
  --partitions 1 --replication-factor 1
```

Napisz strumień, który filtruje transakcje > 5000 PLN i wysyła je do tematu `alerts`.

In [ ]:
from pyspark.sql.functions import to_json, struct

# TWÓJ KOD
# alerts = parsed.filter(col("kwota") > 5000)
# query = (alerts
#     .select(to_json(struct("*")).alias("value"))
#     .writeStream
#     .format("kafka")
#     .option("kafka.bootstrap.servers", "localhost:9092")
#     .option("topic", "alerts")
#     .option("checkpointLocation", "/tmp/checkpoint-alerts")
#     .start()
# )

Sprawdź w terminalu, czy alerty trafiają do Kafki:
```bash
docker compose exec kafka kafka-console-consumer \
  --topic alerts --bootstrap-server localhost:9092 --from-beginning
```

---

## Praca domowa

1. Napisz kompletny pipeline: producent → Kafka → Spark Streaming → agregacja w oknach → zapis do Kafki.
2. Dodaj filtrowanie po sklepie (np. tylko Warszawa).
3. Wyślij kod do repozytorium Git.

Na następnych zajęciach: detekcja anomalii w czasie rzeczywistym.

In [ ]:
spark.stop()